In [1]:
import torch
from torchvision import models, transforms
from PIL import Image
import cv2
import numpy as np
import urllib.request
import time
import os

In [2]:


if not os.path.exists("imagenet_classes.txt"):
    print("Downloading class labels...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        "imagenet_classes.txt"
    )

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]



In [3]:
#MODEL SETUP
print("Loading pre-trained ResNet18 model...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict(image_path):

    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0)

    # TODO 4: Add timing logic here to measure inference latency (in milliseconds)
    # Start timer
    start_time = time.perf_counter()

    with torch.no_grad():
        output = model(input_tensor)

    # End timer
    end_time = time.perf_counter()

    latency_ms = (end_time - start_time) * 1000.0

    probs = torch.nn.functional.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probs, 0)



    return categories[predicted_idx.item()], confidence.item(), latency_ms



Loading pre-trained ResNet18 model...


In [4]:
def simulate_turbidity(img_array):
    """
    Simulate murky water (blur/haze).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    kernel_size = (31, 31)

    # Applying Gaussian blur
    img_array= cv2.GaussianBlur(img_array, kernel_size, 0)
    return img_array


In [5]:
import cv2
import numpy as np

def simulate_color_shift(img_array):
    """
    Simulate depth color loss (attenuate the red channel).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    red_channel = img_array[:, :, 2].astype(np.float32)

    # I am assuming 80% of red light is absorbed
    red_channel = red_channel * 0.2
    img_array[:, :, 2] = np.clip(red_channel, 0, 255).astype(np.uint8)

    return img_array

In [6]:
def simulate_sensor_noise(img_array):
    """
    Simulate low-light digital camera noise.
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    noise = np.random.normal(0, 50, img_array.shape)
    # since there is high noise in murky water ,so taking noise as 50.

    img_array= img_array.astype(np.float32) + noise
    img_array = np.clip(img_array, 0, 255)
    img_array = img_array.astype(np.uint8)
    return img_array

In [7]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "aquarium.jpg"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_aquarium.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_aquarium.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_aquarium.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_aquarium.jpg"),
        ("Color Shift", "test_colorshift_aquarium.jpg"),
        ("Sensor Noise", "test_noise_aquarium.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: eel
Confidence: 0.4804
Latency   : 44.89 ms
------------------------------
Condition : Turbidity
Prediction: brambling
Confidence: 0.0456
Latency   : 14.54 ms
------------------------------
Condition : Color Shift
Prediction: eel
Confidence: 0.6180
Latency   : 13.79 ms
------------------------------
Condition : Sensor Noise
Prediction: boa constrictor
Confidence: 0.1242
Latency   : 14.59 ms
------------------------------


### Result for aquarium

|    Baseline (Clean)    |            Turbidity (Murky)             |            Color Shift (Depth Loss)            |          Sensor Noise (Low Light)          |
|:----------------------:|:----------------------------------------:|:----------------------------------------------:|:------------------------------------------:|
| ![Clean](aquarium.jpg) | ![Turbidity](./test_turbid_aquarium.jpg) | ![Color Shift](./test_colorshift_aquarium.jpg) | ![Sensor Noise](./test_noise_aquarium.jpg) |

In [8]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "jellyfish.jpg"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_jellyfish.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_jellyfish.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_jellyfish.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_jellyfish.jpg"),
        ("Color Shift", "test_colorshift_jellyfish.jpg"),
        ("Sensor Noise", "test_noise_jellyfish.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: jellyfish
Confidence: 0.9949
Latency   : 14.15 ms
------------------------------
Condition : Turbidity
Prediction: goldfish
Confidence: 0.1528
Latency   : 11.19 ms
------------------------------
Condition : Color Shift
Prediction: jellyfish
Confidence: 0.9940
Latency   : 12.74 ms
------------------------------
Condition : Sensor Noise
Prediction: dishrag
Confidence: 0.1610
Latency   : 13.99 ms
------------------------------


### Result for jellyfish

|    Baseline (Clean)     |            Turbidity (Murky)            |           Color Shift (Depth Loss)            |         Sensor Noise (Low Light)          |
|:-----------------------:|:---------------------------------------:|:---------------------------------------------:|:-----------------------------------------:|
| ![Clean](jellyfish.jpg) | ![Turbidity](test_turbid_jellyfish.jpg) | ![Color Shift](test_colorshift_jellyfish.jpg) | ![Sensor Noise](test_noise_jellyfish.jpg) |

In [9]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "set_f20_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_f20_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_f20_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_f20_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_f20_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_f20_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_f20_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: hen-of-the-woods
Confidence: 0.2156
Latency   : 12.63 ms
------------------------------
Condition : Turbidity
Prediction: coral fungus
Confidence: 0.1022
Latency   : 10.79 ms
------------------------------
Condition : Color Shift
Prediction: coral reef
Confidence: 0.3028
Latency   : 8.69 ms
------------------------------
Condition : Sensor Noise
Prediction: coral reef
Confidence: 0.2504
Latency   : 8.69 ms
------------------------------


### Result for set_f20_SESR.png

|      Baseline (Clean)      |              Turbidity (Murky)               |              Color Shift (Depth Loss)              |            Sensor Noise (Low Light)            |
|:--------------------------:|:--------------------------------------------:|:--------------------------------------------------:|:----------------------------------------------:|
| ![Clean](set_f20_SESR.png) | ![Turbidity](./test_turbid_set_f20_SESR.jpg) | ![Color Shift](./test_colorshift_set_f20_SESR.jpg) | ![Sensor Noise](./test_noise_set_f20_SESR.jpg) |

In [10]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "set_f46_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_f46_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_f46_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_f46_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_f46_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_f46_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_f46_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: king crab
Confidence: 0.3312
Latency   : 11.61 ms
------------------------------
Condition : Turbidity
Prediction: tarantula
Confidence: 0.4672
Latency   : 11.06 ms
------------------------------
Condition : Color Shift
Prediction: king crab
Confidence: 0.6450
Latency   : 11.20 ms
------------------------------
Condition : Sensor Noise
Prediction: coral reef
Confidence: 0.4412
Latency   : 8.81 ms
------------------------------


### Result for .png

|      Baseline (Clean)      |              Turbidity (Murky)               |              Color Shift (Depth Loss)              |            Sensor Noise (Low Light)            |
|:--------------------------:|:--------------------------------------------:|:--------------------------------------------------:|:----------------------------------------------:|
| ![Clean](set_f46_SESR.png) | ![Turbidity](./test_turbid_set_f46_SESR.jpg) | ![Color Shift](./test_colorshift_set_f46_SESR.jpg) | ![Sensor Noise](./test_noise_set_f46_SESR.jpg) |

In [11]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "set_o20_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_o20_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_o20_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_o20_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_o20_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_o20_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_o20_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: loggerhead
Confidence: 0.7938
Latency   : 14.52 ms
------------------------------
Condition : Turbidity
Prediction: loggerhead
Confidence: 0.5158
Latency   : 12.87 ms
------------------------------
Condition : Color Shift
Prediction: loggerhead
Confidence: 0.6235
Latency   : 12.49 ms
------------------------------
Condition : Sensor Noise
Prediction: loggerhead
Confidence: 0.6046
Latency   : 14.27 ms
------------------------------


### Result for set_o20_SESR.png

|      Baseline (Clean)      |        Turbidity (Murky)         |        Color Shift (Depth Loss)        |      Sensor Noise (Low Light)      |
|:--------------------------:|:--------------------------------:|:--------------------------------------:|:----------------------------------:|
| ![Clean](set_o20_SESR.png) | ![Turbidity](./test_turbid_set_o20_SESR.jpg) | ![Color Shift](./test_colorshift_set_o20_SESR.jpg) | ![Sensor Noise](./test_noise_set_o20_SESR.jpg) |

In [12]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "set_u106_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_u106_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_u106_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_u106_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_u106_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_u106_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_u106_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: rock beauty
Confidence: 0.8001
Latency   : 13.00 ms
------------------------------
Condition : Turbidity
Prediction: rock beauty
Confidence: 0.2477
Latency   : 12.11 ms
------------------------------
Condition : Color Shift
Prediction: rock beauty
Confidence: 0.8690
Latency   : 12.91 ms
------------------------------
Condition : Sensor Noise
Prediction: rock beauty
Confidence: 0.4675
Latency   : 14.27 ms
------------------------------


### Result for set_u106_SESR.png

|      Baseline (Clean)       |        Turbidity (Murky)         |        Color Shift (Depth Loss)        |      Sensor Noise (Low Light)      |
|:---------------------------:|:--------------------------------:|:--------------------------------------:|:----------------------------------:|
| ![Clean](set_u106_SESR.png) | ![Turbidity](./test_turbid_set_u106_SESR.jpg) | ![Color Shift](./test_colorshift_set_u106_SESR.jpg) | ![Sensor Noise](./test_noise_set_u106_SESR.jpg) |

In [13]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "set_u113_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_u113_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_u113_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_u113_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_u113_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_u113_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_u113_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: king crab
Confidence: 0.2095
Latency   : 10.84 ms
------------------------------
Condition : Turbidity
Prediction: barn spider
Confidence: 0.2184
Latency   : 10.92 ms
------------------------------
Condition : Color Shift
Prediction: leatherback turtle
Confidence: 0.1820
Latency   : 11.15 ms
------------------------------
Condition : Sensor Noise
Prediction: tarantula
Confidence: 0.3480
Latency   : 12.97 ms
------------------------------


### Result for set_u113_SESR.png

|      Baseline (Clean)       |        Turbidity (Murky)         |        Color Shift (Depth Loss)        |      Sensor Noise (Low Light)      |
|:---------------------------:|:--------------------------------:|:--------------------------------------:|:----------------------------------:|
| ![Clean](set_u113_SESR.png) | ![Turbidity](./test_turbid_set_u113_SESR.jpg) | ![Color Shift](./test_colorshift_set_u113_SESR.jpg) | ![Sensor Noise](./test_noise_set_u113_SESR.jpg) |